# Day 1 - Data Ingestion
Bluestock Fintech Capstone | June 2026

Loading all 10 datasets, exploring fund master, fetching live NAV data, validating AMFI codes.


In [ ]:
import pathlib
import pandas as pd
import numpy as np

BASE = pathlib.Path.cwd()
RAW  = BASE / "data" / "raw"


## load all 10 datasets

In [ ]:
files = {
    "fund_master":           "01_fund_master.csv",
    "nav_history":           "02_nav_history.csv",
    "aum_by_fund_house":     "03_aum_by_fund_house.csv",
    "monthly_sip_inflows":   "04_monthly_sip_inflows.csv",
    "category_inflows":      "05_category_inflows.csv",
    "industry_folio_count":  "06_industry_folio_count.csv",
    "scheme_performance":    "07_scheme_performance.csv",
    "investor_transactions": "08_investor_transactions.csv",
    "portfolio_holdings":    "09_portfolio_holdings.csv",
    "benchmark_indices":     "10_benchmark_indices.csv",
}

dfs = {}
for name, fname in files.items():
    df = pd.read_csv(RAW / fname)
    dfs[name] = df
    nulls = df.isnull().sum().sum()
    print(f"{fname:40s}  shape={str(df.shape):18s}  nulls={nulls}")


## fund_master - shape and dtypes

In [ ]:
df_fund = dfs["fund_master"]
print(df_fund.shape)
print(df_fund.dtypes)


In [ ]:
df_fund.head()

## fund master - unique values (task 6)

In [ ]:
print("fund houses:", df_fund["fund_house"].nunique())
print(df_fund["fund_house"].value_counts())


In [ ]:
print("categories:")
print(df_fund["category"].value_counts())
print()
print("sub_categories:")
print(df_fund["sub_category"].value_counts())


In [ ]:
print("risk grades:")
print(df_fund["risk_category"].value_counts())
print()
print("plans:", df_fund["plan"].unique())
print("amfi code range:", df_fund["amfi_code"].min(), "-", df_fund["amfi_code"].max())


## nav_history - basic stats

In [ ]:
df_nav = dfs["nav_history"]
df_nav["date"] = pd.to_datetime(df_nav["date"])

print("shape:", df_nav.shape)
print("date range:", df_nav["date"].min().date(), "to", df_nav["date"].max().date())
print("unique schemes:", df_nav["amfi_code"].nunique())


In [ ]:
# rows per scheme - checking consistency
rows_per = df_nav.groupby("amfi_code").size()
print(rows_per.describe())
# all schemes should have the same number of rows


## amfi code validation (task 7)

In [ ]:
master_codes = set(df_fund["amfi_code"])
nav_codes    = set(df_nav["amfi_code"])

print("fund_master codes:", len(master_codes))
print("nav_history codes:", len(nav_codes))
print("codes in both:", len(master_codes & nav_codes))

missing = master_codes - nav_codes
if missing:
    print("codes missing from nav_history:", sorted(missing))
else:
    print("all fund_master codes are in nav_history - good")

extra = nav_codes - master_codes
if extra:
    print("extra codes in nav_history:", sorted(extra))
else:
    print("no unexpected codes in nav_history")


## live nav files summary (tasks 4 and 5)

In [ ]:
import glob

live = sorted(glob.glob(str(RAW / "live_nav_*.csv")))
# excluding the combined file
live = [f for f in live if "all_schemes" not in f]

print(f"{len(live)} scheme files fetched")
print()

for f in live:
    df_l = pd.read_csv(f, parse_dates=["nav_date"])
    name = pathlib.Path(f).stem
    latest = df_l.iloc[-1]
    print(name)
    print(f"  rows: {len(df_l)}  |  {df_l['nav_date'].min().date()} to {df_l['nav_date'].max().date()}")
    print(f"  latest nav: {latest['nav']:.4f}  ({latest['nav_date'].date()})")
    print()


## observations

- all 10 datasets loaded without errors
- fund_master: 40 schemes, 10 AMCs, 2 categories (equity/debt), 12 sub-categories, 5 risk grades
- nav_history: 46,000 rows, 1,150 rows per scheme, no missing values
- monthly_sip_inflows: 12 null values in yoy_growth_pct column - expected since first 12 months have no prior year
- all 40 amfi codes in fund_master are present in nav_history, no mismatches
- 6 live NAV CSVs saved from mfapi.in (fallback to nav_history used if API is blocked)
- amfi code range is 100016 to 149324, mix of regular and direct plans
